# NLU messages (Phase 1d) — L40 + vLLM

Generates labelled user messages for the 7 intents (`request_diss`, `another_one`, `make_meaner`, `make_softer`, `greet`, `goodbye`, `out_of_scope`). Messages that mention traits come with the exact words that express each trait, which `prepare_nlu.py` turns into BIO slot tags. Messages about held-out traits go to the NLU test set only. About 3,000 messages before cleaning.

**Upload next to this notebook:** `traits.json` from `data/` in the repo.

**Workflow:** `MODE = "pilot"` → Run All → read the samples → `MODE = "full"` → Restart & Run All → copy `llm_out/nlu.jsonl` and `llm_out/nlu_meta.json` to `data/nlu/llm_out/` in the repo → `python data/nlu/prepare_nlu.py`.

**Model:** Qwen/Qwen3-32B-FP8 (Apache-2.0).

## 1. Install

In [ ]:
%pip install -q vllm

## 2. Settings

In [ ]:
import os, re, json, math, time, random, itertools
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # skip FlashInfer's sampler, which needs nvcc >= 12.8 to compile
from collections import Counter, defaultdict
from datetime import date

# ======================= THE SWITCH =======================
MODE = "pilot"              # "pilot" = a few requests to read the output | "full" = the real run
# ==========================================================
assert MODE in ("pilot", "full")

MODEL_ID = "Qwen/Qwen3-32B-FP8"           # same model as the quatrain corpus; thinking switched off below
CHAT_KW  = {"chat_template_kwargs": {"enable_thinking": False}}
SEED, TEMPERATURE, TOP_P = 42, 0.9, 0.95
MAX_TOKENS = 2048

NLU_PER_REQUEST     = 20       # messages per request
NLU_TRAIT_REQUESTS  = 2        # per trait (training and held-out), each in a different style
NLU_PAIRS, NLU_PER_PAIR = 60, 5
NLU_INTENT_REQUESTS = {"request_diss": 8, "another_one": 15, "make_meaner": 15, "make_softer": 15,
                       "greet": 10, "goodbye": 10, "out_of_scope": 15}   # request_diss here = no trait given

OUT_DIR = "llm_out" if MODE == "full" else "llm_pilot_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| output:", OUT_DIR)

## 3. Traits

In [ ]:
TRAITS  = json.load(open("traits.json", encoding="utf-8"))
TRAIN   = [t for t, v in TRAITS.items() if not v["heldout"]]
HELDOUT = [t for t, v in TRAITS.items() if v["heldout"]]
print(f"{len(TRAIN)} training traits | held out: {HELDOUT}")

## 4. Load the model

If it runs out of memory, lower `gpu_memory_utilization` to 0.85.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, max_model_len=4096, gpu_memory_utilization=0.90,
          enable_prefix_caching=True, seed=SEED)
params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS, repetition_penalty=1.05)

def chat(msgs):
    # one reply per conversation -> (texts, output tokens)
    if not msgs:
        return [], 0
    outs = llm.chat(msgs, params, use_tqdm=True, **CHAT_KW)
    return [o.outputs[0].text for o in outs], sum(len(o.outputs[0].token_ids) for o in outs)

## 5. Prompts

Trait messages mix direct mentions with indirect descriptions, and every request uses one of six writing styles so the messages are not all alike.

In [ ]:
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})

def normalize(text):
    return text.translate(QUOTE_MAP)        # curly quotes -> straight, as in form_checks.py

THINK_RE = re.compile(r"<think>.*?</think>", re.S)

def parse_json_array(text):
    m = re.search(r"\[.*\]", THINK_RE.sub("", text), re.S)
    try:
        v = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None
    return v if isinstance(v, list) else None

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

STYLES = ["casual texting: lowercase, slang and a typo or two", "polite, complete sentences",
          "very short, 2 to 6 words", "Gen Z slang", "rambling, with extra details about their day",
          "written by a non-native English speaker"]
NLU_SYSTEM = "You write realistic chat messages that users send to a chatbot. Reply with a JSON array only."
INTENT_DESC = {
    "request_diss": "ask the bot to roast them, without saying anything about themselves",
    "another_one":  "already got a roast and ask for another one",
    "make_meaner":  "already got a roast and ask for a harsher, more savage one",
    "make_softer":  "already got a roast and ask for a gentler, nicer one",
    "greet":        "greet the bot at the start of a chat",
    "goodbye":      "end the chat or say goodbye",
    "out_of_scope": "ask the roast bot for something unrelated, like the weather, homework help, a recipe or a fact",
}

def nlu_trait_prompt(traits, k, style):
    desc = " and ".join(f'"{t}" (e.g. {", ".join(TRAITS[t]["synonyms"])})' for t in traits)
    return (f"Write {k} different messages a user might send to a roast-poem chatbot, asking to be roasted "
            f"and revealing that they are {desc}. Each message is about the user themself (\"I\", \"me\").\n"
            f"Style: {style}.\n"
            'Mix direct mentions ("roast me, I\'m super organized") with indirect ones that only describe the '
            'trait ("people say I never shut up about history"). Do not always use the words given above.\n'
            'Return a JSON array of objects: {"text": the message, "spans": [{"trait": '
            + " or ".join(f'"{t}"' for t in traits)
            + ', "span": the exact words in the text that express it}]}. Copy every span exactly from the text.')

def trait_name(label, traits):
    # the model often labels a span with a synonym ("clever") instead of the trait ("smart")
    label = str(label).strip().lower()
    for t in traits:
        if label == t or label in (w.lower() for w in TRAITS[t]["synonyms"]):
            return t
    return traits[0] if len(traits) == 1 else label

def nlu_intent_prompt(intent, k, style):
    return (f"Write {k} different messages where a user of a roast-poem chatbot would {INTENT_DESC[intent]}.\n"
            f"Style: {style}.\nReturn a JSON array of strings.")

print(nlu_trait_prompt(["knowledgeable"], NLU_PER_REQUEST, STYLES[0]))

## 6. Run

In [ ]:
def nlu_jobs():
    rng, jobs = random.Random(SEED + 2), []
    msg = lambda p: [{"role": "system", "content": NLU_SYSTEM}, {"role": "user", "content": p}]
    trait_list = TRAIN[:2] + HELDOUT[:1] if MODE == "pilot" else TRAIN + HELDOUT
    for t in trait_list:
        for style in rng.sample(STYLES, NLU_TRAIT_REQUESTS):
            jobs.append(("request_diss", [t], style, msg(nlu_trait_prompt([t], NLU_PER_REQUEST, style))))
    for p in rng.sample(list(itertools.combinations(TRAIN, 2)), 2 if MODE == "pilot" else NLU_PAIRS):
        style = rng.choice(STYLES)
        jobs.append(("request_diss", list(p), style, msg(nlu_trait_prompt(list(p), NLU_PER_PAIR, style))))
    for intent, n in NLU_INTENT_REQUESTS.items():
        for _ in range(1 if MODE == "pilot" else n):
            style = rng.choice(STYLES)
            jobs.append((intent, [], style, msg(nlu_intent_prompt(intent, NLU_PER_REQUEST, style))))
    return jobs

jobs, t0 = nlu_jobs(), time.time()
print(f"{len(jobs)} requests")
texts, tokens = chat([m for *_, m in jobs])
nlu_rows, bad = [], Counter()
for (intent, traits, style, _), text in zip(jobs, texts):
    items = parse_json_array(text)
    if items is None:
        bad["bad_json"] += 1
        continue
    for it in items:
        if traits:
            if not (isinstance(it, dict) and isinstance(it.get("text"), str)):
                bad["bad_item"] += 1; continue
            spans = [{"trait": trait_name(s.get("trait"), traits), "span": normalize(str(s.get("span") or ""))}
                     for s in it.get("spans") or [] if isinstance(s, dict)]
            text_, group = it["text"], "heldout" if any(TRAITS[t]["heldout"] for t in traits) else "train"
        else:
            if not isinstance(it, str):
                bad["bad_item"] += 1; continue
            text_, spans, group = it, [], "train"
        nlu_rows.append({"intent": intent, "text": normalize(text_).strip(), "spans": spans,
                         "traits": traits, "group": group, "style": style})
dump(f"{OUT_DIR}/nlu.jsonl", nlu_rows)
stats = {"messages": len(nlu_rows), "by_intent": dict(Counter(r["intent"] for r in nlu_rows)),
         "problems": dict(bad), "tokens": tokens, "seconds": round(time.time() - t0, 1)}
print(f"{len(nlu_rows):,} messages | {stats['by_intent']} | problems {dict(bad)}")

## 7. Samples

Check that the spans really are the words that express the trait, and that the messages sound like real users.

In [ ]:
for r in random.Random().sample(nlu_rows, min(12, len(nlu_rows))):
    print(f"  [{r['intent']}] {r['text']}   {r['spans'] or ''}")

## 8. Save the run record

In [ ]:
meta = {"date": date.today().isoformat(), "mode": MODE, "model": MODEL_ID, "license": "Apache-2.0",
        "temperature": TEMPERATURE, "top_p": TOP_P, "seed": SEED,
        "settings": {"NLU_PER_REQUEST": NLU_PER_REQUEST, "NLU_TRAIT_REQUESTS": NLU_TRAIT_REQUESTS, "NLU_PAIRS": NLU_PAIRS,
                     "NLU_PER_PAIR": NLU_PER_PAIR, "NLU_INTENT_REQUESTS": NLU_INTENT_REQUESTS, "STYLES": STYLES},
        "prompts": {"system": NLU_SYSTEM, "trait_example": nlu_trait_prompt(["confident"], NLU_PER_REQUEST, STYLES[0]),
                    "intent_example": nlu_intent_prompt("make_meaner", NLU_PER_REQUEST, STYLES[0])},
        "stats": stats}
json.dump(meta, open(f"{OUT_DIR}/nlu_meta.json", "w", encoding="utf-8"), indent=2)
print("Saved to", OUT_DIR)